# ⚡🌾 VAJRA Platform GPU Benchmarking Notebook

Welcome to the official **VAJRA (Vectorized Analytics for Joint Resilience & Allocation)** GPU Benchmarking Notebook. This notebook runs on a Google Colab GPU instance to benchmark and compare pipeline runs on **CPU vs NVIDIA GPU** for both the **GRID** (power distribution) and **AGRO** (smallholder agriculture) decision systems.

NVIDIA RAPIDS cuDF, XGBoost-GPU, and CuPy are utilized to prove matched acceleration scale-by-scale.

### Step 1: Extract Code and Install Dependencies
Upload `vajra-submission-kit.zip` to Colab files, then run the cell below to unzip and install requirements.

In [ ]:
!unzip -q vajra-submission-kit.zip
!pip install -r vajra/requirements.txt

### Step 2: Verify the NVIDIA GPU VM Environment
Run `nvidia-smi` to verify the active Colab accelerator (T4, L4, or A100 GPU).

In [ ]:
!nvidia-smi

### Step 3: Run the GRID Pipeline Benchmark Matrix
Benchmark scale comparison (1M, 10M, and 50M rows) for the grid telemetry pipeline.

In [ ]:
!cd vajra && python -m bench.run_bench --domain grid --scales 1e6,10e6,50e6 --engines cpu,gpu

### Step 4: Run the AGRO Pipeline Benchmark Matrix
Benchmark scale comparison (1M, 10M, and 50M rows) for the Sentinel-2 NDVI agricultural pipeline.

In [ ]:
!cd vajra && python -m bench.run_bench --domain agro --scales 1e6,10e6,50e6 --engines cpu,gpu

### Step 5: Generate Large GPU Production Artifacts
Generate full-scale 100M row production datasets and compile optimized GPU pipeline runs.

In [ ]:
# GRID - 100M Row GPU pipeline run
!cd vajra && python -m data_gen.generate --meters 200000 --days 6 --out data/city_100m
!cd vajra && python -m pipeline.run --data data/city_100m --engine gpu --out out/gpu_run

# AGRO - 96M Row GPU pipeline run
!cd vajra && python -m agro.generate --plots 4000000 --out data/agro_100m
!cd vajra && python -m agro.run --data data/agro_100m --engine gpu --out out/agro_gpu

### Step 6: Save Results
Copy the benchmark output files into the api packaging directories so the web dashboard serves them.

In [ ]:
!mkdir -p vajra/api/data vajra/api/data_agro
!cp -v vajra/out/gpu_run/*.json vajra/api/data/
!cp -v vajra/out/agro_gpu/*.json vajra/api/data_agro/

# Zip results for download
!zip -r vajra_gpu_results.zip vajra/bench/results.json vajra/api/data/ vajra/api/data_agro/